# UCI BNN — Aggregated Results

Loads all saved `.pt` runs across splits for each dataset, computes RMSE, NLL, and CRPS
on the held-out test set, then produces a paper-ready table (mean ± SEM across splits).

In [ ]:
from __future__ import annotations

import math
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from torch import Tensor
import os
if Path.cwd().name == "notebooks":
    os.chdir("..")
from sazz.scripts.bnns.uci_bnn import (
    load_raw_datasets, make_split, build_target, build_target_learned_noise,
    BNNConfig, BASE_SEED,
)
from sazz.models.neural_networks import FFN
from sazz.utils.bnn_utils import ParamSpec

In [ ]:
RESULTS_DIR = Path("results/uci_bnn")
DATASETS    = ["energy"]#[boston, "naval", "energy"]
N_SPLITS    = 5

# Pretty display names for the table
SAMPLER_LABELS = {
    # "zigzag":           "ZigZag",
    # "sticky_zigzag":    "Sticky ZigZag",
    # "boomerang":        "Boomerang",
    # "sticky_boomerang": "Sticky Boomerang",
    # "nuts":             "NUTS",
    # "nuts_horseshoe":   "NUTS-HS",
    "zigzag_ln":           "ZigZag (LN)",
    "sticky_zigzag_ln":    "Sticky ZigZag (LN)",
    "boomerang_ln":        "Boomerang (LN)",
    "sticky_boomerang_ln": "Sticky Boomerang (LN)",
    "nuts_ln":             "NUTS (LN)",
    "nuts_horseshoe_ln":   "NUTS-HS (LN)",
    #"nuts_unfair_ln":             "unfair NUTS (LN)",
    #"nuts_horseshoe_unfair_ln":   "unfair NUTS-HS (LN)",
}

## Prediction helpers

In [ ]:
def get_noise_std(run: dict, samples: Tensor) -> float:
    """Return per-sample noise std for use in metrics.
    For learned-noise runs, noise_std is exp(samples[:, -1]).
    """
    sampler = run["sampler"]
    if sampler.endswith("_ln") or "_ln" in str(run.get("pipeline", "")):
        return None  # signal to caller: use per-sample sigma
    return run["noise_std"]


def predict_from_samples(
    samples: Tensor,
    target,
    X_test: Tensor,
    learned_noise: bool = False,
) -> tuple[Tensor, Tensor, float]:
    """Returns (mean_pred, epistemic_std, noise_std).

    For learned-noise models the returned noise_std is the posterior mean
    of sigma = exp(log_sigma).
    """
    likelihood = target.meta["model"].likelihood
    preds = torch.stack([
        likelihood.predict(beta, X_test).squeeze(-1) for beta in samples
    ])  # [S, N]
    mean_pred = preds.mean(0)    # [N]
    epist_std = preds.std(0)     # [N]

    if learned_noise:
        noise_std = float(samples[:, -1].exp().mean())
    else:
        noise_std = float(likelihood.noise_std)

    return mean_pred, epist_std, noise_std

## Metrics

In [ ]:
def rmse(y_true: Tensor, mean_pred: Tensor, y_std: float) -> float:
    return float(((mean_pred - y_true) ** 2).mean().sqrt()) * y_std


def nll(y_true: Tensor, mean_pred: Tensor, epist_std: Tensor,
        noise_std: float, y_std: float) -> float:
    total_std = (epist_std ** 2 + noise_std ** 2).sqrt()
    ll = (
        -0.5 * ((y_true - mean_pred) / total_std) ** 2
        - total_std.log()
        - 0.5 * math.log(2 * math.pi)
    ).mean()
    # convert to original scale
    return float(-ll + math.log(y_std))


def crps_gaussian(y_true: Tensor, mean_pred: Tensor, epist_std: Tensor,
                  noise_std: float, y_std: float) -> float:
    """Closed-form CRPS for a Gaussian predictive, on the original scale."""
    from torch.distributions import Normal
    sigma = (epist_std ** 2 + noise_std ** 2).sqrt() * y_std
    mu    = mean_pred * y_std
    yt    = y_true * y_std
    d     = Normal(0.0, 1.0)
    z     = (yt - mu) / sigma
    crps  = sigma * (z * (2 * d.cdf(z) - 1) + 2 * d.log_prob(z).exp() - 1 / math.sqrt(math.pi))
    return float(crps.mean())


def compute_metrics(y_true, mean_pred, epist_std, noise_std, y_std) -> dict:
    return {
        "RMSE": rmse(y_true, mean_pred, y_std),
        "NLL":  nll(y_true, mean_pred, epist_std, noise_std, y_std),
        "CRPS": crps_gaussian(y_true, mean_pred, epist_std, noise_std, y_std),
    }

## Load all runs and compute metrics

In [ ]:
print("Loading raw datasets for test-set reconstruction...")
raw = load_raw_datasets()
print("Done.")

In [ ]:
records = []  # list of dicts: {dataset, sampler, split_id, RMSE, NLL, CRPS}

for dataset in DATASETS:
    ds_dir = RESULTS_DIR / dataset
    if not ds_dir.exists():
        print(f"  [{dataset}] no results directory found, skipping.")
        continue

    if dataset not in raw:
        print(f"  [{dataset}] not available in raw data, skipping.")
        continue

    X_all, y_all = raw[dataset]

    for split_id in range(N_SPLITS):
        split_dir = ds_dir / f"split_{split_id:02d}"
        if not split_dir.exists():
            continue

        data = make_split(X_all, y_all, seed=BASE_SEED + split_id)
        X_test = data["X_test"]
        y_test = data["y_test"]
        y_std  = data["y_std"]

        # Cache targets so we don't rebuild for every .pt in the same split
        target_cache: dict[bool, object] = {}

        for pt_path in sorted(split_dir.glob("*.pt")):
            run = torch.load(pt_path, map_location="cpu", weights_only=False)
            sampler = run["sampler"]
            wall_time = run["elapsed_sec"]
            suffix  = "_ln" if pt_path.stem.endswith("_ln") else ""
            key     = pt_path.stem           # e.g. "sticky_boomerang_ln"
            learned = suffix == "_ln"

            if learned not in target_cache:
                cfg = BNNConfig(
                    layer_sizes=run["layer_sizes"],
                    activation=run["activation"],
                    noise_std=run["noise_std"],
                    learned_noise=learned,
                )
                builder = build_target_learned_noise if learned else build_target
                target_cache[learned] = builder(data, cfg)

            target = target_cache[learned]
            samples = run["samples"]  # [S, D]

            try:
                mean_pred, epist_std, noise_std = predict_from_samples(
                    samples, target, X_test, learned_noise=learned
                )
                m = compute_metrics(y_test, mean_pred, epist_std, noise_std, y_std)
                records.append({
                    "dataset":  dataset,
                    "sampler":  key,
                    "split_id": split_id,
                    **m,
                    "wall time": wall_time,
                })
                print(f"  [{dataset} / split {split_id} / {key}]  "
                      f"RMSE={m['RMSE']:.3f}  NLL={m['NLL']:.3f}  CRPS={m['CRPS']:.3f}")
            except Exception as e:
                print(f"  [{dataset} / split {split_id} / {key}] ERROR: {e}")

df = pd.DataFrame(records)
print(f"\nTotal records: {len(df)}")

## Aggregate: mean ± SEM across splits

In [ ]:
def mean_sem(x):
    return x.mean(), x.sem()

rows = []
for (dataset, sampler), g in df.groupby(["dataset", "sampler"]):
    row = {"Dataset": dataset, "Sampler": SAMPLER_LABELS.get(sampler, sampler)}
    for metric in ["RMSE", "NLL", "CRPS", "wall time"]:
        mu, sem = mean_sem(g[metric])
        if metric == "wall time":
            mu_min = mu/60
            if sem >= 60:
                mu_min += sem//60
                sem_min = sem%60
                row[metric] = f"{mu_min:.1f}min ± {sem_min:.1f}s"
            else:
                row[metric] = f"{mu_min:.1f}min ± {sem:.1f}s"
        else:
            row[metric] = f"{mu:.5f} ± {sem:.3f}"
    rows.append(row)

table = pd.DataFrame(rows).set_index(["Dataset", "Sampler"])
table

## LaTeX export

In [ ]:
print(table.to_latex(escape=False))